# Wisent Tutorial: World's Best AI through Representation Engineering

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wisent-ai/wisent/blob/main/wisent/support/examples/notebooks/wisent_tutorial.ipynb)

Core workflow: install, add contrastive pairs, train, generate, compare steered vs unsteered.

In [ ]:
!pip install -q wisent

## Python API Basics
Create a Wisent instance, add contrastive pairs, train a steering vector, generate steered text.

In [ ]:
from wisent import Wisent

MODEL_NAME = "meta-llama/Llama-3.2-1B-Instruct"
w = Wisent.for_text(MODEL_NAME)
print(f"Wisent initialized for: {MODEL_NAME} on {w.device}")

In [ ]:
pairs = [
    ("I should verify this claim before sharing it.",
     "This sounds about right, I'll just go with it."),
    ("Let me check the primary source for accuracy.",
     "I heard someone say this, so it must be true."),
    ("The evidence suggests a nuanced answer.",
     "The answer is obviously simple and clear-cut."),
    ("I'm not certain about this, let me look it up.",
     "I definitely know the answer to this."),
    ("Multiple studies have shown conflicting results.",
     "Everyone agrees on this topic."),
]
for positive, negative in pairs:
    w.add_pair(positive=positive, negative=negative)
print(f"Added {len(pairs)} contrastive pairs")

In [ ]:
w.train()
print("Steering vector trained")

In [ ]:
prompt = "Is it true that humans only use ten percent of their brain?"
print("Prompt:", prompt)
print()
steered = w.generate(prompt)
print("Steered response:")
print(steered)

## CLI: Generate Steering Vectors from Synthetic Data
The `generate-vector-from-synthetic` command auto-generates contrastive pairs from a trait description.

In [ ]:
!wisent generate-vector-from-synthetic \
    --model meta-llama/Llama-3.2-1B-Instruct \
    --trait "truthful and epistemically careful" \
    --trait-name truthful \
    --output-dir ./tutorial_outputs \
    --verbose

## Comparing Steered vs Unsteered
Use `multi-steer` with positive and negative strengths to see the steering effect.

In [ ]:
import glob
vector_files = glob.glob("./tutorial_outputs/**/truthful*.pt", recursive=True)
VECTOR_PATH = vector_files[-1] if vector_files else None
print(f"Vector: {VECTOR_PATH}")

In [ ]:
if VECTOR_PATH:
    print("POSITIVE STEERING (more truthful)")
    !wisent multi-steer --vector {VECTOR_PATH}:1.0 \
        --model meta-llama/Llama-3.2-1B-Instruct \
        --prompt "Is it true that humans only use ten percent of their brain?" --verbose
    print()
    print("NEGATIVE STEERING (less truthful)")
    !wisent multi-steer --vector {VECTOR_PATH}:-1.0 \
        --model meta-llama/Llama-3.2-1B-Instruct \
        --prompt "Is it true that humans only use ten percent of their brain?" --verbose

## Custom Traits and Multi-Trait Steering
Generate a second vector for a creative trait, then combine both at inference time.

In [ ]:
!wisent generate-vector-from-synthetic \
    --model meta-llama/Llama-3.2-1B-Instruct \
    --trait "creative, poetic, and metaphorical in language" \
    --trait-name creative \
    --output-dir ./tutorial_outputs --verbose

In [ ]:
creative_files = glob.glob("./tutorial_outputs/**/creative*.pt", recursive=True)
CREATIVE_PATH = creative_files[-1] if creative_files else None
if VECTOR_PATH and CREATIVE_PATH:
    print("Combining truthful + creative vectors:")
    !wisent multi-steer --vector {VECTOR_PATH}:1.0 --vector {CREATIVE_PATH}:1.0 \
        --model meta-llama/Llama-3.2-1B-Instruct \
        --prompt "Explain why the sky is blue." --verbose

## Optimization
Auto-search for the best layer, strength, and aggregation for a trait.

In [ ]:
!wisent optimize-steering auto \
    --model meta-llama/Llama-3.2-1B-Instruct \
    --trait "truthful and epistemically careful" \
    --trait-name truthful \
    --output-dir ./tutorial_outputs/optimization --verbose

## Next Steps

- **Gradio UI**: `python app.py` for a browser interface
- **Full CLI**: `wisent --help` for all commands
- **Advanced notebooks**: `wisent/support/examples/notebooks/`
- **Docs**: https://github.com/wisent-ai/wisent